In [2]:
import numpy as np, pandas as pd, joblib
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, classification_report

df = pd.read_csv("../data/processed/features_2022_v2.csv")
df["block"] = (df.lat // 0.1).astype(int) * 100 + (df.lon // 0.1).astype(int)

FEATS = ["elevation", "slope", "dist_water_m", "hand", "elev_mean_1km", "elev_rel",
         "cropland", "builtup", "wetland_mangrove"]
y = (df.flood_frac >= 0.10).astype(int)

# out-of-fold predictions: har cell ka score us model se jo us block ko dekh hi nahi paya
oof = np.zeros(len(df))
for tr, te in GroupKFold(5).split(df, y, df.block):
    m = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                      scale_pos_weight=(y.iloc[tr] == 0).sum() / (y.iloc[tr] == 1).sum())
    m.fit(df.iloc[tr][FEATS], y.iloc[tr])
    oof[te] = m.predict_proba(df.iloc[te][FEATS])[:, 1]

print("Blocked OOF AUC:", round(roc_auc_score(y, oof), 3))
print(classification_report(y, (oof > 0.5).astype(int), digits=2))

# final model poore data pe
final = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                      scale_pos_weight=(y == 0).sum() / (y == 1).sum())
final.fit(df[FEATS], y)
joblib.dump({"model": final, "features": FEATS}, "../models/susceptibility_v1.pkl")

df["risk_score"] = final.predict_proba(df[FEATS])[:, 1]
df["risk_level"] = pd.cut(df.risk_score, [0, 0.33, 0.66, 1.0], labels=["Low", "Medium", "High"], include_lowest=True)
df[["cell_id", "lat", "lon", "risk_score", "risk_level"]].to_csv("../data/processed/risk_map_2022.csv", index=False)
print(df.risk_level.value_counts())

Blocked OOF AUC: 0.76
              precision    recall  f1-score   support

           0       0.95      0.69      0.80      4932
           1       0.23      0.71      0.35       668

    accuracy                           0.69      5600
   macro avg       0.59      0.70      0.57      5600
weighted avg       0.86      0.69      0.74      5600

risk_level
Low       2702
Medium    1759
High      1139
Name: count, dtype: int64


In [3]:
q = df.risk_score.rank(pct=True)
df["risk_level"] = pd.cut(q, [0, 0.70, 0.90, 1.0], labels=["Low", "Medium", "High"], include_lowest=True)
print(df.risk_level.value_counts())
print(df.groupby("risk_level", observed=True).flooded_actual.mean() if "flooded_actual" in df else
      df.assign(f=y).groupby("risk_level", observed=True).f.mean().round(3))
df[["cell_id", "lat", "lon", "risk_score", "risk_level"]].to_csv("../data/processed/risk_map_2022.csv", index=False)

risk_level
Low       3920
Medium    1120
High       560
Name: count, dtype: int64
risk_level
Low       0.028
Medium    0.234
High      0.529
Name: f, dtype: float64


In [1]:
import numpy as np, pandas as pd
feat = pd.read_csv("../data/processed/features_2022_v2.csv")
risk = pd.read_csv("../data/processed/risk_map_2022.csv")

# cell ka area (sq km) latitude ke hisaab se, 100 m pixels = 100 per sq km
area_km2 = (111.32 * 0.01) * (111.32 * np.cos(np.radians(feat.lat)) * 0.01)
feat["pop_est"] = (feat.population.fillna(0) * area_km2 * 100).round()

out = risk.merge(feat[["cell_id", "pop_est", "flood_frac"]], on="cell_id")
out.to_csv("../data/processed/risk_map_2022.csv", index=False)

print("Total population (AOI):", int(out.pop_est.sum()))
print(out.groupby("risk_level", observed=True).pop_est.sum().astype(int))
print("Aug 2022 flood mein estimated log:", int((out.pop_est * out.flood_frac).sum()))

Total population (AOI): 3450431
risk_level
High       343213
Low       2387543
Medium     719675
Name: pop_est, dtype: int64
Aug 2022 flood mein estimated log: 183188
